# 04 · A/B test: design (REAL base rates) and readout (SIMULATED)

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 80)
from scipy.stats import chisquare, norm
import numpy as np

## Sample size by hand for +5 % on purchase conversion (alpha 5 %, power 80 %)

In [2]:
base = json.loads((ROOT / 'data/public/ab_design/base_rates.json').read_text())
p1 = base['conversion']; p2 = p1 * 1.05; pbar = (p1 + p2) / 2
za, zb = norm.ppf(0.975), norm.ppf(0.80)
n = (za * np.sqrt(2 * pbar * (1 - pbar)) + zb * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2 / (p2 - p1) ** 2
round(p1, 5), int(np.ceil(n))

(0.00799, 798587)

In [3]:
pd.read_csv(ROOT / 'data/public/ab_design/mde_after_3_weeks.csv')

,metric,traffic,n_per_arm_3w,mde_3w
0,Engaged-visitor rate (2+ views),real,111661,0.020495
1,Engaged-visitor rate (2+ views),sim,1000000,0.006998
2,Views per visitor,real,111661,0.023494
3,Views per visitor,sim,1000000,0.007998
4,Add-to-cart rate,real,111661,0.074482
5,Add-to-cart rate,sim,1000000,0.024494
6,Purchase conversion,real,111661,0.136966
7,Purchase conversion,sim,1000000,0.044989


## SRM: chi-square on the split, before and after excluding day 9

In [4]:
srm = pd.read_csv(ROOT / 'data/simulated/ab_readout/srm_summary.csv')
srm['p_check'] = [chisquare([c, t]).pvalue for c, t in zip(srm.control, srm.treatment)]
srm

,stage,control,treatment,share_treatment,p_value,p_check
0,all visitors,999412,988916,0.497361,9.801882e-14,9.801882e-14
1,excluding day(s) [9],951631,952778,0.500301,4.058847e-01,4.058847e-01


## Results, novelty and decision

In [5]:
res = pd.read_csv(ROOT / 'data/simulated/ab_readout/results.csv')
res[['metric', 'control', 'treatment', 'rel_lift', 'rel_ci_low', 'rel_ci_high', 'p_value']]

,metric,control,treatment,rel_lift,rel_ci_low,rel_ci_high,p_value
0,Feed clicks per visitor (primary),1.670167,2.008202,0.202396,0.195012,0.209780,0.000000e+00
1,Engaged-visitor rate (2+ clicks),0.276121,0.304084,0.101274,0.096372,0.106175,0.000000e+00
2,Add-to-cart rate,0.025806,0.027114,0.050692,0.032583,0.068801,1.865468e-08
3,Purchase conversion,0.008096,0.008366,0.033412,0.001184,0.065639,3.885444e-02
4,Revenue per visitor (EUR),0.368353,0.370684,0.006329,-0.036550,0.049207,7.716731e-01


In [6]:
pd.read_csv(ROOT / 'data/simulated/ab_readout/novelty.csv')

,week,rel_lift,rel_ci_low,rel_ci_high,active_control,active_treatment
0,1,0.290837,0.278364,0.303310,333498,334471
1,2,0.194899,0.184295,0.205503,397426,398604
2,3,0.153534,0.144767,0.162302,541969,541445


In [7]:
json.loads((ROOT / 'data/simulated/ab_readout/decision.json').read_text())['rules']

{'SRM resolved (p >= 0.001 after documented exclusion)': True,
 'Primary metric up, p < 0.05': True,
 'Guardrails within margin': True,
 'Conversion not harmed (CI low > -2 %)': True}

## Business case (public pricing, simulated customer)

In [8]:
pd.read_csv(ROOT / 'data/simulated/business_case/cases.csv')

,case,conversion_lift,incremental_gmv,incremental_revenue,annual_cost,net_benefit,roi_multiple,payback_months
0,low (CI lower bound),0.001184,1.480488e+05,17765.852049,314001.6,-296235.747951,0.056579,NaN
1,central (point estimate),0.033412,4.176469e+06,501176.259092,314001.6,187174.659092,1.596095,1.282225
2,high (CI upper bound),0.065639,8.204889e+06,984586.666135,314001.6,670585.066135,3.135610,0.357896
